In [1]:
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
#check role


role = sagemaker.get_execution_role()

print(role)

arn:aws:iam::998923791081:role/LabRole


In [3]:
#check your bucket


s3 = boto3.client("s3")

response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-998923791081


In [ ]:
#package up a trained model to deploy it  AWS SageMaker
#because deploying our model to cloud platforms like AWS SageMaker, Google Cloud Vertex AI, or Azure ML, 
#their systems are hardcoded to look for a single compressed archive.

import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "artifacts/credit_score_pipeline.pkl",
        arcname="model_credit_score.pkl"
    )

In [10]:
#Upload model to S3, because deployment will take the model form s3

s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-998923791081"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [11]:
#check model can be run or not

from inference import model_fn

model = model_fn("model")

In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel


# ---- EDIT THESE ---------------------------------------------------------
BUCKET = "sagemaker-us-east-1-998923791081"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )
    
    print("Testing endpoint with sample data...")
    sample = {
        "instances": [[5.1, 3.5, 1.4, 0.2]]}
    
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=str(sample).replace("'", '"'),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))

    #print(
     #   f"\nEndpoint '{ENDPOINT_NAME}' is live in {REGION}.\n"
      #  f"Delete it before lab teardown: predictor.delete_endpoint()"
    #)


if __name__ == "__main__":
    main()


Role:      arn:aws:iam::998923791081:role/LabRole
Model URI: s3://sagemaker-us-east-1-998923791081/model/model.tar.gz
Endpoint:  iris-endpoint

Deploying endpoint (5-8 minutes)...
-----!Testing endpoint with sample data...

Smoke test response:
{"probabilities": [[1.0, 0.0, 0.0]], "predictions": [0], "labels": ["iris-setosa"]}


In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?utm_source=chatgpt.com&region=us-east-1#logsV2:log-groups

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

Deleting failed endpoint: iris-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: iris-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete! You can now safely run your main deploy script.
